# Jalon A - Smoke Demo

Ce notebook montre le flux minimal: conventions -> activations -> probe.

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from boardGPT.nn import GPT, GPTConfig
from boardGPT.interpretability import (
    get_othello_move_mappings,
    move_ids_to_notation,
    move_ids_to_board_labels,
    extract_residuals,
    LinearBoardProbe,
    fit_probe,
    eval_probe,
)

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

In [ ]:
move_to_id, id_to_move = get_othello_move_mappings()
sample_moves = ['d3', 'c3', 'c4', 'e3']
sample_ids = [0] + [move_to_id[m] for m in sample_moves]

print('ids:', sample_ids)
print('notation:', move_ids_to_notation(sample_ids))
board = move_ids_to_board_labels(sample_ids)
print('board labels len:', len(board), 'classes:', sorted(set(board)))

In [ ]:
cfg = GPTConfig(block_size=8, vocab_size=20, n_layer=2, n_head=2, n_embd=16, dropout=0.0, bias=True)
model = GPT(cfg, use_flash=False)

tokens = torch.randint(0, cfg.vocab_size, (12, cfg.block_size))
residuals = extract_residuals(model, tokens, layer=1, device='cpu')
print('residuals shape:', tuple(residuals.shape))

In [ ]:
# Jeu de donnees synthetique lineairement decodable
x = torch.randn(256, 16)
w = torch.randn(16, 64, 3)
logits = torch.einsum('nd,dck->nck', x, w)
y = logits.argmax(dim=-1)

train_loader = DataLoader(TensorDataset(x[:192], y[:192]), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x[192:], y[192:]), batch_size=32, shuffle=False)

probe = LinearBoardProbe(input_dim=16)
history = fit_probe(probe, train_loader, val_loader=val_loader, epochs=8, learning_rate=5e-3, device='cpu')
stats = eval_probe(probe, val_loader, device='cpu')

print('last train loss:', history['train_loss'][-1])
print('val acc:', stats['acc_global'])

Si ces cellules tournent, l'integration Jalon A est fonctionnelle.